In [13]:
from dotenv import load_dotenv
import json
from groq import Groq
import os
import requests
from PyPDF2 import PdfReader
import gradio as gr

c:\Users\sanoj\ed donner\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
!pip install gradio


  Using cached gradio-5.49.1-py3-none-any.whl.metadata (16 kB)
  Using cached aiofiles-24.1.0-py3-none-any.whl.metadata (10 kB)
  Using cached Brotli-1.1.0-cp312-cp312-win_amd64.whl.metadata (5.6 kB)
  Using cached fastapi-0.119.1-py3-none-any.whl.metadata (28 kB)
  Using cached ffmpy-0.6.3-py3-none-any.whl.metadata (2.9 kB)
  Using cached gradio_client-1.13.3-py3-none-any.whl.metadata (7.1 kB)
  Using cached groovy-0.1.2-py3-none-any.whl.metadata (6.1 kB)
  Using cached huggingface_hub-0.35.3-py3-none-any.whl.metadata (14 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached markupsafe-3.0.3-cp312-cp312-win_amd64.whl.metadata (2.8 kB)
  Using cached orjson-3.11.3-cp312-cp312-win_amd64.whl.metadata (43 kB)
  Using cached pandas-2.3.3-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached pillow-11.3.0-cp312-cp312-win_amd64.whl.metadata (9.2 kB)
  Using cached pydantic-2.11.10-py3-none-any.whl.metadata (68 kB)
  Using cached pydub-0.25.1-py2.py3-none-any.w

In [3]:
!pip install numpy

  Using cached numpy-2.3.4-cp312-cp312-win_amd64.whl.metadata (60 kB)
Using cached numpy-2.3.4-cp312-cp312-win_amd64.whl (12.8 MB)


In [14]:
import numpy

print("NumPy version:", numpy.__version__)


NumPy version: 2.3.4


In [15]:
load_dotenv(override=True)
groq=Groq()

GroqError: The api_key client option must be set either by passing api_key to the client or by setting the GROQ_API_KEY environment variable

In [3]:
pushover_user=os.getenv("PUSHOVER_USER")
pushover_token=os.getenv("PUSHOVER_TOKEN")
pushover_url="https://api.pushover.net/1/messages.json"

In [4]:
def push(message):
    print(f"push:{message}")
    payload = {"user":pushover_user,"token":pushover_token,"message":message}
    requests.post(pushover_url,data=payload)

In [5]:
push("hi")

push:hi


In [ ]:
def record_user_details(email,name="name not provided",notes="not provided"):
    push(f"Recording {name} with email {email} and notes {notes}")
    return{"recording":"ok"}

In [30]:
def record_unknown_question(question):
    push(f"Recording {question} ,sorry i couldnt answering")
    return {"recording":"ok"}

In [9]:
record_user_details_json = {
    "name":"record_user_details",
    "description":"use this tool to record that a user email",
    "parameters":{
        "type":"object",
        "properties":{
            "email":{
                "type":"string",
                "decritption":"the email address of the user"
            },
            "name":{
                "type":"string",
                "description":"the user name"
            },
            "notes":{
                "type":"string",
                "description":"any additional notes"
            }
        }
    },
    "required":["email"],
    "additionalPropertises":False
}

In [11]:
unknown_user_question_json = {
    "name":"record_unknown_question",
    "description":"always use this tool to record any unknown question that couldnt be answer",
    "parameters":{
        "question":{
            "type":"string",
            "description":"the question that couldnt be answered"
        }
    },
    "required":["question"],
    "additionalProperties":False
}

In [12]:
tools=[{"type":"function", "function":record_user_details_json},
{"type":"function" ,"function":unknown_user_question_json}]

In [13]:
tools

[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'use this tool to record that a user email',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'decritption': 'the email address of the user'},
     'name': {'type': 'string', 'description': 'the user name'},
     'notes': {'type': 'string', 'description': 'any additional notes'}}},
   'required': ['email'],
   'additionalPropertises': False}},
 {'type': 'function',
  'function': {'name': 'record_unknown_question',
   'description': 'always use this tool to record any unknown question that couldnt be answer',
   'parameters': {'question': {'type': 'string',
     'description': 'the question that couldnt be answered'}},
   'required': ['question'],
   'additionalProperties': False}}]

In [ ]:
def handle_tool_call(tool_calls):
    results=[]
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments= json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name} ")
        
        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        
        elif tool_name == "record_unknown_question":
            result =record_unknown_question(**arguments)

        results.append({"role":"tools","content":json.dumps(result),"tool_call_id":tool_call.id})
    return results


In [36]:
globals()["record_unknown_question"]("what is this")

push:Recording what is this ,sorry i couldnt answering


{'recording': 'ok'}

In [38]:
from PyPDF2 import PdfReader

reader = PdfReader("sanoj/sanojcsam_resume.pdf")
resume=""
for page in reader.pages:
    text = page.extract_text()
    if text:
        resume+=text



with open("sanoj/summary.txt" ,"r" ,encoding="utf-8") as file:
    summary = file.read()



name = "sanoj c sam"


In [39]:
system_prompt = f"""you are acting as {name}. you are answering question on {name},
please give answer to question related to {name}'s career,background,skills ,experience
you are given a summary {name} bckground and resume {resume} which you can answer the questions,
ifyou dont know the answer,say sorry,i dont know,first time only greeting"""

system_prompt += f"f##summary:{summary} \n  profile:{resume} \n"

In [45]:
def chat(message,history):
    messages = [{"role":"system","content":system_prompt}] +  [clean_message(m) for m in history] + [{"role": "user", "content": message}]   
    response = groq.chat.completions.create(model="openai/gpt-oss-20b",
     messages=messages,)
    return response.choices[0].message.content

In [46]:
from groq import Groq

def chat(message,history):
    messgaes =[{"role":"system", "content":system_prompt}] + history + [{"role":"user","content":message}]
    done =False
    while not done:

        response = groq.chat.completions.create(model="openai/gpt-oss-20b",messages=messages ,tools=tools) 

        final = response.choices[0].final

        if final == "tool_calls":
            message =response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_call(tool_calls)
            messages.append(message)
            messages.extend(results)
        
        else:
            done =True

    return messages.choices[0].message.content
